In [10]:
"""
This performs a rate of rise cut across all available bands for the candidates in
sn_near_galaxy_candidate.csv, extragalactic_lt20mag_candidates.csv and
candidates_in_TNS.csv. It pulls the diaObjectIds from the CSV file, fetches the light
curve data from the Fink API, computes a peak-normalised pre-peak linear slope for
every band present in each object's light curve, and passes any object with at least
one band showing a nonzero pre-peak rise. This replaces the earlier g/r-only, fixed
empirical threshold version, since that was found to miss objects whose only clear
pre-peak rise was in a band other than g or r (e.g. i-band for higher redshift SNe).
"""

import pandas as pd
import numpy as np
from pathlib import Path
from light_curve_plotter import collect_light_curve_data

# Directories for the objects that passed the first filtering done through the tags, second directory is for the output of the rate of rise cut.
TEST_DIR = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data"
PLOTS_DIR = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "images" / "test_data_images"
# Reading the csv for the objects that passed the Fink tag filtering and creating the CSVs where data will be stored.

# Specifying how many objects go into each query to the API.
CHUNK_SIZE = 500

# Minimum number of pre-peak points needed in a band before a slope is even attempted.
# np.polyfit needs at least 2 points to fit a line; kept low deliberately while we test
# the loosened nonzero-slope criterion, per today's discussion with Anais.
MIN_PRE_PEAK_POINTS = 2

# Minimum time baseline (in days) spanned by the pre-peak points used in a fit.
# Points clustered within a very short window can produce wildly unstable slopes,
# since the denominator of the fit (variance in time) collapses toward zero even
# with many points - this was seen directly in some objects showing slopes of 10-100+
# despite having 10-20+ pre-peak detections, all crammed into a single night or two.
MIN_PRE_PEAK_TIME_SPAN_DAYS = 2


def collect_light_curve_info_with_fallback(candidates: list[str]) -> pd.DataFrame | None:
    """This function collects the lightcurve info of an object
    and factors in a recursive self repair incase the script timesout
    while running on large requests."""

    data = collect_light_curve_data(",".join(candidates))
    if data is not None:
        return data

    if len(candidates) <= 1:
        print(f"Could not fetch data for object {candidates}, skipping.")
        return None

    mid = len(candidates) // 2
    left = collect_light_curve_info_with_fallback(candidates[:mid])
    right = collect_light_curve_info_with_fallback(candidates[mid:])

    candidates_parts = [df for df in (left, right) if df is not None]
    if not candidates_parts:
        return None
    return pd.concat(candidates_parts)


def chunked_light_curve_data(candidates: list, chunk_size: int = 500) -> pd.DataFrame:
    """
    This function scrapes the Fink API for the light curve info necessary
    to make cuts based on object qualities and produce plots of the light curves.
    """
    candidate_results = []

    for diaObjectId in range(0, len(candidates), chunk_size):
        chunked_ids = candidates[diaObjectId:diaObjectId + chunk_size]
        candidates_light_curve_data = collect_light_curve_info_with_fallback(chunked_ids)
        if candidates_light_curve_data is not None:
            candidate_results.append(candidates_light_curve_data)

    if not candidate_results:
        raise RuntimeError(f"No light curve data returned for candidate batch of size {len(candidates)}")

    return pd.concat(candidate_results, ignore_index=True)


def slope_calculation(group_df: pd.DataFrame, band: str, min_points: int = MIN_PRE_PEAK_POINTS, min_time_span: int = MIN_PRE_PEAK_TIME_SPAN_DAYS) -> tuple[float, int] | None:
    """
    This function finds the slope of the band of the lightcurve prepeak by fitting a
    polynomial of degree 1 to the points of the light curve before the brightest point
    of the light curve. Returns None if there isn't enough pre-peak data in this band
    to attempt a fit.

    This is done to remove the flat objects whose brightness doesn't change that much
    over time. We make it a function of bands so that we can easily use different
    bandpasses when necessary for higher redshift SNe.
    """
    band_df = group_df[group_df["r:band"] == band]

    if band_df.empty:
        return None

    peak_flux = band_df["r:psfFlux"].max()
    peak_mjd = band_df.loc[band_df["r:psfFlux"].idxmax(), "r:midpointMjdTai"]
    pre_peak = band_df[band_df["r:midpointMjdTai"] < peak_mjd]

    if len(pre_peak) < min_points:
        return None

    
    time_span = pre_peak["r:midpointMjdTai"].max() - pre_peak["r:midpointMjdTai"].min()
    if time_span < min_time_span:
        return None

    normalised_flux = pre_peak["r:psfFlux"] / peak_flux
    normalised_err = pre_peak["r:psfFluxErr"] / peak_flux

    # Guard against zero/negative errors, which would otherwise produce an
    # infinite or undefined weight.
    normalised_err = normalised_err.clip(lower=1e-6)
    weights = 1 / normalised_err

    slope, _ = np.polyfit(pre_peak["r:midpointMjdTai"], normalised_flux, 1, w=weights)


    return slope, len(pre_peak)


def compute_all_band_slopes(group_df: pd.DataFrame, min_points: int = MIN_PRE_PEAK_POINTS) -> dict[str, tuple[float, int]]:
    """
    Computes the pre-peak slope for every band present in this object's light curve,
    not just a fixed pair. Returns a dict of band -> (slope, n_points) for every band
    where a slope could be calculated; bands with insufficient pre-peak data are
    omitted rather than included as None, so downstream code doesn't need to filter.
    """
    bands_present = group_df["r:band"].unique()
    slopes = {}

    for band in bands_present:
        result = slope_calculation(group_df, band, min_points)
        if result is not None:
            slopes[band] = result

    return slopes


def check_rate_of_rise(group_df: pd.DataFrame, min_points: int = MIN_PRE_PEAK_POINTS) -> bool:
    """
    Computes the pre-peak slope across every available band for this object, and
    passes the object if at least one band shows a nonzero (positive) pre-peak rise.

    This is deliberately loose for now: rather than requiring specific bands (as the
    old g/r-only version did) or a calibrated threshold, this just checks whether the
    object is doing *anything* pre-peak in at least one band. Tighter criteria
    (uncertainty weighting, max time-gap between points, a proper threshold) are
    intended to be layered on top of this once it's validated on a small batch.
    """
    slopes = compute_all_band_slopes(group_df, min_points)

    if not slopes:
        return False

    return any(slope > 0 for slope, _ in slopes.values())


def applying_the_rate_of_rise(objects: list[str], chunk_size: int, output_path_passed: Path, output_path_failed: Path) -> tuple[list[str], list[str]]:
    """
    This function pulls the light curve data for the specified list of object ids and
    the chunk size can be customised if you have a large list of objects or small. The
    function then checks whether the object shows a nonzero pre-peak rise in at least
    one band and directs the objects to their respective lists. Finally this function
    produces two CSV files of the objects that passed and ones that failed.
    """
    passed, failed = [], []

    light_curve_info = chunked_light_curve_data(objects, chunk_size)

    for object_id, group_df in light_curve_info.groupby("r:diaObjectId"):
        if check_rate_of_rise(group_df):
            passed.append(object_id)
        else:
            failed.append(object_id)

    pd.DataFrame(passed, columns=["r:diaObjectId"]).to_csv(output_path_passed, index=False)
    pd.DataFrame(failed, columns=["r:diaObjectId"]).to_csv(output_path_failed, index=False)

    return passed, failed


# if __name__ == "__main__":
#     applying_the_rate_of_rise(SN_NEAR_GALAXY_CANDIDATES, CHUNK_SIZE, SN_NEAR_GALAXY_CANDIDATES_OUTPUT_PASSED, SN_NEAR_GALAXY_CANDIDATES_OUTPUT_FAILED)
#     applying_the_rate_of_rise(EXTRAGALACTIC_CANDIDATES, CHUNK_SIZE, EXTRAGALACTIC_CANDIDATES_OUTPUT_PASSED, EXTRAGALACTIC_CANDIDATES_OUTPUT_FAILED)
#     applying_the_rate_of_rise(IN_TNS_CANDIDATES, CHUNK_SIZE, IN_TNS_CANDIDATES_OUTPUT_PASSED, IN_TNS_CANDIDATES_OUTPUT_FAILED)

test_candidates = pd.read_csv("/home/riley/Documents/University/Honours/Honours_Project/data/test_data/test_objects.csv", dtype={"r:diaObjectId":str})["r:diaObjectId"].tolist()

applying_the_rate_of_rise(test_candidates, 500, "/home/riley/Documents/University/Honours/Honours_Project/data/test_data/test_passed.csv", "/home/riley/Documents/University/Honours/Honours_Project/data/test_data/test_failed.csv")

([170019696305242310,
  170019696442081285,
  170019716323082348,
  170019716324130933,
  170019716327800983,
  170019716479320072,
  170019717135204495,
  170028485639143520,
  170028485639143548,
  170028485642813476,
  170028485648056327,
  170032900274651160,
  170032917163016282,
  170046089928376424,
  170235219334922248,
  170261592629837872,
  313699514821640259,
  313835836426682441,
  313945770546430083,
  313963405286309936],
 [])

In [11]:
import io
import requests as re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.time import Time

SLOPES_OUTPUT_PATH = Path.home() / "Documents" / "University" / "Honours" / "Honours_Project" / "data" / "test_data" / "test_fitted_slopes.csv"

def append_fitted_slopes(rows: list[dict], output_path: Path = SLOPES_OUTPUT_PATH) -> None:
    """
    Appends per-band fitted slope rows to a CSV file, writing a header only if the
    file doesn't already exist. Each row is expected to contain diaObjectId, band,
    slope, and n_pre_peak_points.
    """
    if not rows:
        return

    new_rows_df = pd.DataFrame(rows)

    new_rows_df.to_csv(output_path, mode="a", header=True, index=False)

band_colours = {"u": "darkblue", "g":"blue", "r": "red", "i":"darkred", "z":"orange", "y":"black"}   

def collect_light_curve_data(diaObjectId):
    r = re.post(
        "https://api.lsst.fink-portal.org/api/v1/sources",
        json={
            "diaObjectId": diaObjectId,
            "columns": "r:diaObjectId, r:diaSourceId, r:midpointMjdTai, r:psfFlux, r:psfFluxErr, r:band",
            "output-format": "json",
        },
    )
    if r.status_code != 200:
        print (f"Failed to fetch the light curve info for {diaObjectId}: {r.status_code}")
        return None
    return pd.read_json(io.BytesIO(r.content))

def lightcurve_plotting(df: pd.DataFrame, diaObjectId: str, source_file: str, output_dir: str, min_pre_peak_points: int = 2, min_pre_peak_time_span: float = 2.0, slopes_output_path: Path = SLOPES_OUTPUT_PATH):
    """
    Plots the light curve for one object directly in flux space (no magnitude
    conversion), with the pre-peak linear fit for each band overlaid as a dashed
    line. Also prints the fitted slope and number of pre-peak points used, per band,
    and appends the same values to slopes_output_path so they can be inspected after
    a batch run rather than just read off the console.

    Working in flux rather than magnitude avoids fitting a straight line to a
    logarithmic quantity, which was flagged as a potential source of distortion in
    the rate-of-rise filter's output.
    """
    df = df[df["r:psfFlux"] > 0].copy()
    df = df.sort_values("r:midpointMjdTai")
    df["date"] = Time(df["r:midpointMjdTai"].values, format="mjd").to_datetime()

    fig, ax = plt.subplots(figsize=(10, 6))
    fitted_slope_rows = []

    for band, group in df.groupby("r:band"):
        colour = band_colours.get(band, "black")

        ax.errorbar(
            group["date"],
            group["r:psfFlux"],
            yerr=group["r:psfFluxErr"],
            fmt="o",
            color=colour,
            label=f"{band}-band",
            capsize=3,
            markersize=5,
        )

        # Fit and overlay the pre-peak slope for this band
        peak_flux = group["r:psfFlux"].max()
        peak_mjd = group.loc[group["r:psfFlux"].idxmax(), "r:midpointMjdTai"]
        pre_peak = group[group["r:midpointMjdTai"] < peak_mjd]

        if len(pre_peak) < min_pre_peak_points:
            print(f"  [{diaObjectId}] {band}-band: only {len(pre_peak)} pre-peak point(s), skipping fit.")
            continue

        time_span = pre_peak["r:midpointMjdTai"].max() - pre_peak["r:midpointMjdTai"].min()
        if time_span < min_pre_peak_time_span:
            print(f"  [{diaObjectId}] {band}-band: pre-peak points only span {time_span:.2f} days, skipping fit.")
            continue

        normalised_flux = pre_peak["r:psfFlux"] / peak_flux
        normalised_err = (pre_peak["r:psfFluxErr"] / peak_flux).clip(lower=1e-6)
        weights = 1 / normalised_err

        slope, intercept = np.polyfit(pre_peak["r:midpointMjdTai"], normalised_flux, 1, w=weights)

        print(f"  [{diaObjectId}] {band}-band: slope={slope:.5f}, n_pre_peak={len(pre_peak)}")

        fitted_slope_rows.append({
            "diaObjectId": diaObjectId,
            "band": band,
            "slope": slope,
            "n_pre_peak_points": len(pre_peak),
        })

        fit_mjd = pre_peak["r:midpointMjdTai"].sort_values()
        fit_flux = (slope * fit_mjd + intercept) * peak_flux
        fit_dates = Time(fit_mjd.values, format="mjd").to_datetime()

        ax.plot(
            fit_dates,
            fit_flux,
            linestyle="--",
            color=colour,
            alpha=0.7,
            label=f"{band} fit (slope={slope:.4f})",
        )

    ax.set_xlabel("Observation Date")
    ax.set_ylabel("Flux (nanojansky)")
    ax.set_title(f"Light curve - {diaObjectId}\n({source_file})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig(f"{output_dir}.png")
    plt.close(fig)

    append_fitted_slopes(fitted_slope_rows, slopes_output_path)

In [13]:
# Batch plotter for the labelled in_tns objects. This was done so that I can understand what the light curves of a SN Ia looks like. Builds a validation dataset.
import pandas as pd
from pathlib import Path

def batch_plotter(object_ids, chunk_size=500, output_path=Path.home() / "/home/riley/Documents/University/Honours/Honours_Project/images/SN_Ia_lightcurves/passed_rate_of_rise_cuts_results"):
    for diaObjectId in range(0, len(object_ids), chunk_size):
        chunk = ",".join(object_ids[diaObjectId:diaObjectId + chunk_size])
        light_curve_info = collect_light_curve_data(chunk)
        if light_curve_info is None or light_curve_info.empty:
            print(f"No light curve data found for chunk starting at index {diaObjectId}.")
            continue
        for object_id, group_df in light_curve_info.groupby("r:diaObjectId"):
            object_id = str(object_id)
            # lightcurve_plotting takes a df, diaObjectId and a source file string as arguments
            lightcurve_plotting(group_df, object_id, object_id, output_path / f"{object_id}")
batch_plotter(test_candidates, 500, Path.home() / "Documents"/"University"/"Honours"/"Honours_Project"/"images"/"test_data_images"/"overlayed_plots")

  [170019696305242310] g-band: slope=0.01029, n_pre_peak=15
  [170019696305242310] i-band: only 0 pre-peak point(s), skipping fit.
  [170019696305242310] r-band: slope=0.02132, n_pre_peak=6
  [170019696442081285] g-band: pre-peak points only span 0.01 days, skipping fit.
  [170019696442081285] i-band: slope=-0.00446, n_pre_peak=63
  [170019696442081285] r-band: slope=0.00936, n_pre_peak=27
  [170019696442081285] z-band: pre-peak points only span 1.01 days, skipping fit.
  [170019716323082348] g-band: slope=0.06553, n_pre_peak=174
  [170019716323082348] i-band: slope=0.05941, n_pre_peak=361
  [170019716323082348] r-band: slope=0.05631, n_pre_peak=185
  [170019716323082348] y-band: slope=0.03242, n_pre_peak=5
  [170019716323082348] z-band: slope=0.04646, n_pre_peak=166
  [170019716324130933] g-band: pre-peak points only span 0.00 days, skipping fit.
  [170019716324130933] i-band: pre-peak points only span 0.00 days, skipping fit.
  [170019716324130933] r-band: slope=0.00244, n_pre_peak=2